[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/your_notebook.ipynb)

In [ ]:
# install (run once)
!pip install datasets transformers[torch] peft evaluate accelerate

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import evaluate

# --- Step 1: Prepare your dataset ---
# We add trust_remote_code=True to resolve the RuntimeError
dataset = load_dataset(
    "parquet",
    data_files={
        "train": "hf://datasets/PolyAI/banking77@refs/convert/parquet/default/train/0000.parquet",
        "test":  "hf://datasets/PolyAI/banking77@refs/convert/parquet/default/test/0000.parquet",
    },
)


df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])

# Split dataset into training (70%), validation (15%), and test (15%)
# We use the training data to create a validation set for monitoring performance
train_data, val_data = train_test_split(df_train, test_size=0.15, random_state=42)
test_data = df_test

print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")

# Tokenize data using a compatible tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(texts):
    # Ensure the text is prepared for input into the model
    return tokenizer(list(texts), padding="max_length", truncation=True, return_tensors="pt")

# Convert to dictionary format for the Trainer
def prepare_ds(df):
    tokenized = tokenize_function(df['text'])
    return [{"input_ids": ids, "attention_mask": mask, "labels": lbl}
            for ids, mask, lbl in zip(tokenized["input_ids"], tokenized["attention_mask"], df['label'])]

train_dataset = prepare_ds(train_data)
val_dataset = prepare_ds(val_data)
test_dataset = prepare_ds(test_data)

In [ ]:
# --- Step 2: Fine-tune the pretrained model (Applying PEFT) ---
# Load pretrained BERT model with 77 labels for Banking77
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=77)

# PEFT Approach: Freeze all layers except the task-specific classification head
for param in model.base_model.parameters():
    param.requires_grad = False

# Optional: Unfreeze the last two encoder layers for better adaptation
for param in model.base_model.encoder.layer[-2:].parameters():
    param.requires_grad = True

# Set up training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,                 # Number of passes
    per_device_train_batch_size=16,     # Batch size
     eval_strategy="epoch",        # Evaluate at end of each epoch
    learning_rate=5e-5                  # Optimized learning rate
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Start fine-tuning using PEFT
trainer.train()

In [ ]:
# --- Step 3: Monitor and evaluate performance ---
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

pred_out = trainer.predict(test_dataset)
preds  = np.argmax(pred_out.predictions, axis=-1)
labels = pred_out.label_ids

accuracy  = accuracy_score(labels, preds)
f1        = f1_score(labels, preds, average="weighted")
precision = precision_score(labels, preds, average="weighted", zero_division=0)
recall    = recall_score(labels, preds, average="weighted", zero_division=0)

print(f"Test Accuracy:  {accuracy:.4f}")
print(f"Test F1 Score:  {f1:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")


In [ ]:
# --- Step 4: Optimize PEFT for your task ---
# To optimize, you could adjust the learning_rate to 2e-5 or unfreeze 3 layers